# 부록 09. Streaming으로 실시간 응답 처리하기

LangGraph의 스트리밍 기능을 활용하여 에이전트 실행을 실시간으로 모니터링하는 방법을 학습합니다.

## 학습 목표
- 다양한 스트리밍 모드 이해 (updates, values, messages)
- 서브그래프 스트리밍
- 커스텀 스트리밍 데이터 전송

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. 스트리밍 모드 개요

LangGraph는 5가지 스트리밍 모드를 지원합니다:

| 모드 | 설명 | 용도 |
|------|------|------|
| `updates` | 각 노드 실행 후 상태 업데이트 | 노드별 진행 상황 |
| `values` | 각 노드 실행 후 전체 상태 | 현재 상태 확인 |
| `messages` | LLM 토큰 스트림 | 실시간 텍스트 출력 |
| `custom` | 사용자 정의 데이터 | 커스텀 진행 상황 |
| `debug` | 상세 디버그 정보 | 디버깅 |

## 3. 기본 스트리밍: updates 모드

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 조회합니다."""
    return f"{city}: 맑음, 22°C"

@tool
def get_time() -> str:
    """현재 시간을 반환합니다."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_weather, get_time],
    system_prompt="당신은 도움이 되는 어시스턴트입니다."
)

print("에이전트 생성 완료")

In [ ]:
# updates 모드: 각 노드의 상태 변경 확인
print("=== updates 모드 스트리밍 ===")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]},
    stream_mode="updates"
):
    for node_name, value in chunk.items():
        print(f"\n📍 노드: {node_name}")
        
        if "messages" in value:
            for msg in value["messages"]:
                msg_type = msg.__class__.__name__
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    print(f"   → 도구 호출: {[tc['name'] for tc in msg.tool_calls]}")
                elif hasattr(msg, 'content'):
                    content = msg.content[:100] if isinstance(msg.content, str) else str(msg.content)[:100]
                    print(f"   → {msg_type}: {content}")

## 4. values 모드: 전체 상태 확인

In [ ]:
# values 모드: 매 단계마다 전체 상태
print("=== values 모드 스트리밍 ===")

step = 0
for state in agent.stream(
    {"messages": [{"role": "user", "content": "현재 시간 알려줘"}]},
    stream_mode="values"
):
    step += 1
    print(f"\n📊 Step {step} - 메시지 수: {len(state.get('messages', []))}")
    
    # 마지막 메시지만 출력
    if state.get("messages"):
        last_msg = state["messages"][-1]
        msg_type = last_msg.__class__.__name__
        content = last_msg.content if isinstance(last_msg.content, str) else str(last_msg.content)
        print(f"   마지막 메시지 ({msg_type}): {content[:80]}..." if len(content) > 80 else f"   마지막 메시지 ({msg_type}): {content}")

## 5. 복합 모드: updates + values

In [ ]:
# 여러 모드 동시 사용
print("=== 복합 모드 스트리밍 ===")

final_state = None
for mode, chunk in agent.stream(
    {"messages": [{"role": "user", "content": "서울 날씨와 현재 시간 알려줘"}]},
    stream_mode=["updates", "values"]
):
    if mode == "updates":
        for node_name, value in chunk.items():
            print(f"🔄 [updates] 노드 '{node_name}' 실행됨")
    elif mode == "values":
        final_state = chunk
        msg_count = len(chunk.get('messages', []))
        print(f"📊 [values] 현재 상태: {msg_count}개 메시지")

print(f"\n최종 응답: {final_state['messages'][-1].content}")

## 6. 서브그래프 스트리밍

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from langchain.chat_models import init_chat_model

# 상태 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 서브그래프 정의
def create_sub_agent():
    model = init_chat_model("gpt-4o-mini", temperature=0)
    
    def process(state: AgentState) -> dict:
        response = model.invoke(state["messages"])
        return {"messages": [response]}
    
    sub_graph = StateGraph(AgentState)
    sub_graph.add_node("process", process)
    sub_graph.add_edge(START, "process")
    sub_graph.add_edge("process", END)
    return sub_graph.compile()

# 메인 그래프
sub_agent = create_sub_agent()

def route_to_sub(state: AgentState) -> dict:
    return {"messages": [HumanMessage(content="서브에이전트로 위임됨")]}

def call_sub_agent(state: AgentState) -> dict:
    result = sub_agent.invoke(state)
    return result

main_graph = StateGraph(AgentState)
main_graph.add_node("router", route_to_sub)
main_graph.add_node("sub_agent", call_sub_agent)
main_graph.add_edge(START, "router")
main_graph.add_edge("router", "sub_agent")
main_graph.add_edge("sub_agent", END)

main_app = main_graph.compile()

print("서브그래프가 있는 메인 그래프 생성 완료")

In [ ]:
# subgraphs=True로 서브그래프 스트리밍
print("=== 서브그래프 스트리밍 ===")

for namespace, mode, chunk in main_app.stream(
    {"messages": [HumanMessage(content="안녕하세요!")]},
    stream_mode=["updates", "values"],
    subgraphs=True
):
    graph_name = namespace if len(namespace) > 0 else "main"
    
    if mode == "updates":
        for node_name, value in chunk.items():
            print(f"🔄 [{graph_name}] → [{node_name}]")

## 7. 비동기 스트리밍

In [ ]:
# 비동기 스트리밍
print("=== 비동기 스트리밍 ===")

async def stream_agent():
    async for chunk in agent.astream(
        {"messages": [{"role": "user", "content": "날씨 알려줘"}]},
        stream_mode="updates"
    ):
        for node_name, value in chunk.items():
            print(f"📍 노드: {node_name}")

# Jupyter에서 실행
await stream_agent()

## 8. 실용적인 스트리밍 헬퍼 함수

In [ ]:
def stream_with_progress(agent, query: str, config=None):
    """진행 상황을 표시하며 에이전트 실행"""
    print(f"🚀 쿼리: {query}")
    print("-" * 50)
    
    final_state = None
    tool_calls = []
    
    for mode, chunk in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode=["updates", "values"],
        config=config
    ):
        if mode == "updates":
            for node_name, value in chunk.items():
                if node_name == "agent":
                    # 에이전트 응답 확인
                    if "messages" in value:
                        for msg in value["messages"]:
                            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                                for tc in msg.tool_calls:
                                    print(f"🔧 도구 호출: {tc['name']}")
                                    tool_calls.append(tc['name'])
                elif node_name == "tools":
                    print(f"✅ 도구 실행 완료")
                    
        elif mode == "values":
            final_state = chunk
    
    print("-" * 50)
    print(f"📊 사용된 도구: {tool_calls if tool_calls else '없음'}")
    print(f"\n💬 최종 응답:")
    print(final_state["messages"][-1].content)
    
    return final_state

# 테스트
result = stream_with_progress(agent, "서울 날씨와 현재 시간 알려줘")

## 9. 요약

### 이 노트북에서 배운 것

1. **스트리밍 모드**
   - `updates`: 노드별 상태 변경
   - `values`: 전체 상태 스냅샷
   - `messages`: LLM 토큰 스트림
   - 복합 모드: 여러 모드 동시 사용

2. **서브그래프 스트리밍**
   - `subgraphs=True` 옵션
   - namespace로 그래프 식별

3. **비동기 스트리밍**
   - `astream()` 메서드
   - `async for` 루프

4. **실용적인 패턴**
   - 진행 상황 표시
   - 도구 호출 추적
   - 최종 상태 보존

### 다음 단계
- 부록_10: LangSmith로 Tracing과 디버깅